# Data Ingestion

In [ ]:
# import all necessary libraries
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
from zipfile import ZipFile

In [ ]:
path= r"C:\Users\User\Desktop\new_work\Customer_Behaviour\archive.zip"

In [ ]:
with ZipFile(path, "r") as f:
    f.printdir()

In [ ]:
with ZipFile(path, "r") as f:
    with f.open("ecommerce_customer_behavior_dataset.csv") as file:
        df= pd.read_csv(file)

In [ ]:
df.head(5)

# Premilary Data Analysis

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

# Exploratory Data Analysis

Descriptive Data Analysis

In [ ]:
df.info

In [ ]:
df.describe().T

In [ ]:
categorical_col= df.select_dtypes(include='object').columns

In [ ]:
numerical_col = df.select_dtypes(include='number').columns


In [ ]:
numerical_col.nunique()

In [ ]:
numerical_col

In [ ]:
for col in numerical_col:
    df[col].value_counts()
    print(df[col].value_counts())
    print("="*30)
    print()

In [ ]:
for col in categorical_col:
    df[col].value_counts()
    print(df[col].value_counts())
    print("="*30)
    print()

In [ ]:
df= df.drop(columns=['Order_ID'])

In [ ]:
df= df.drop(columns=['Customer_ID'])

In [ ]:
df= df.drop(columns=['Date'])

In [ ]:
# Exploratory Data Analysis

# Let's examine the numeric features only for some analyses
numeric_df = df.select_dtypes(include=[np.number])

# If there are four or more numeric columns, display a correlation heatmap
if numeric_df.shape[1] >= 4:
    plt.figure(figsize=(12, 8))
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='viridis')
    plt.title('Correlation Heatmap of Numeric Features')
    plt.show()
else:
    print('Not enough numeric columns to generate a correlation heatmap.')

# Plotting distributions for some key numeric variables
numeric_columns = ['Age', 'Unit_Price', 'Quantity', 'Discount_Amount', 'Total_Amount', 
                   'Session_Duration_Minutes', 'Pages_Viewed', 'Delivery_Time_Days', 'Customer_Rating']

plt.figure(figsize=(14, 10))
for i, col in enumerate(numeric_columns):
    plt.subplot(3, 3, i+1)
    sns.histplot(df[col].dropna(), kde=True, color='skyblue')
    plt.title(f'Distribution of {col}')
    plt.tight_layout()
plt.show()

# Count plot for categorical variable: Payment_Method
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='Payment_Method', palette='pastel')
plt.title('Distribution of Payment Methods')
plt.xticks(rotation=45)
plt.show()

# Pair plot for a subset of features to visually inspect relationships
subset_features = ['Age', 'Total_Amount', 'Session_Duration_Minutes', 'Pages_Viewed', 'Customer_Rating']
sns.pairplot(df[subset_features].dropna(), diag_kind='kde', corner=True)
plt.show()

# Data Preprocessing

In [ ]:
df

In [ ]:
Payment_Method= df["Payment_Method"].unique()
Payment_Method


In [ ]:
Device_Type= df["Device_Type"].unique()
Device_Type

In [ ]:
# df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

In [ ]:
df.head(1)

In [ ]:
from sklearn.calibration import LabelEncoder
encoder = LabelEncoder()



encoded_labels = encoder.fit_transform(df["Product_Category"])
print("Original labels:", encoded_labels)

df["Product_Category"]= encoder.fit_transform(df["Product_Category"])


In [ ]:
df

In [ ]:
Device_Type= df["Device_Type"].unique()

df["Device_Type"]= encoder.fit_transform(df["Device_Type"])

In [ ]:
Payment_Method= df["Payment_Method"].unique()

df["Payment_Method"] = encoder.fit_transform(df["Payment_Method"])

In [ ]:
City= df["City"].unique()

df["City"]= encoder.fit_transform(df["City"])

In [ ]:
df

In [ ]:

# from sklearn.preprocessing import OrdinalEncoder


# Product_Category = [["Sports","Electronics","Fashion","Beauty","Home & Garden" ,"Food","Books","Toys"]]


# order= LabelEncoder()
# df['Product_Category']=order.fit_transform(df[['Product_Category']])


In [ ]:
from sklearn.preprocessing import OneHotEncoder


Gender= [['Male', 'Female', 'Other']]

ohe= OneHotEncoder(categories=Gender, sparse_output=False, handle_unknown='ignore')
encoded_data=ohe.fit_transform(df[["Gender"]])

encoded_df = pd.DataFrame(
    encoded_data,
    columns=ohe.get_feature_names_out(['Gender'])
)

encoded_df.index = df.index
df = pd.concat([df.drop(columns=['Gender']), encoded_df], axis=1)

In [ ]:
df

In [ ]:
# Converting the boolean target to integer
df['Is_Returning_Customer'].astype(int)

In [ ]:
from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split


x= df.drop(columns=["Is_Returning_Customer"])
y= df["Is_Returning_Customer"]
scaler = StandardScaler()


x[numerical_col]= scaler.fit_transform(x[numerical_col])
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=1, stratify=y)


In [ ]:
# let scale the numerical columns using minmax scaler 
from sklearn.preprocessing import MinMaxScaler

scaler=MinMaxScaler()
x[numerical_col]= scaler.fit_transform(x[numerical_col])

# let scale the numerical columns in test data too
df[numerical_col]=scaler.transform(df[numerical_col])

In [ ]:
print("shape of Training set : ", x_train.shape)
print("Shape of test set : ", x_test.shape)
print("Percentage of classes in training set:")
print(y_train.value_counts(normalize=True))
print("Percentage of classes in test set:")
print(y_test.value_counts(normalize=True))

In [ ]:
df.columns

In [ ]:
# initializing and train Logistic Regression model 
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix


#Preparing the data
features = ['Age', 'Unit_Price', 'Quantity', 'Discount_Amount', 'Total_Amount', 
            'Session_Duration_Minutes', 'Pages_Viewed', 'Delivery_Time_Days', 'Customer_Rating']


model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)


# Making predictions
y_pred = model.predict(x_test)

# Evaluating the model performance using Accruracy Score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of the Returning customer Predictor: {accuracy:.3f}")


# Displaying a detailed classification report
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

# Creating confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('predicted')
plt.ylabel('Actual')
plt.show()


# # Calculating and plotting permutation importance
# result = permutation_importance(model, x_test, y_test, n_repeats=10, random_state=42)
# importance_df = pd.DataFrame({
#     'Feature': features,
#     'Importance': result.importances_mean
# }).sort_values(by='Importance', ascending=True)

# plt.figure(figsize=(8, 6))
# plt.barh(importance_df['numerical_col'], importance_df['Importance'], color='teal')
# plt.title('Permutation Importance of Features')
# plt.xlabel('Mean Importance')
# plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
models = {
        'Logistic Regression': LogisticRegression(),
        'Random Forest': RandomForestClassifier(),
        'KNN': KNeighborsClassifier(),
        'SVM': SVC()
    }

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix


results={}
for model_name, model in models.items():
    print(f'Training {model_name}...')
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[model_name] = accuracy
    print(f'{model_name} Accuracy: {accuracy}')

    # Include the confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)

    # plot with model name as title
    disp.plot(cmap='Blues')
    plt.title(f'confusion_matrix: {model_name}')
    plt.show()


In [ ]:
#Evaluate the model using accuracy , precision, Recall , F1-score and a Confusion Matrix
from sklearn.metrics import classification_report

Best_Model = RandomForestClassifier()
Best_Model.fit(x_train, y_train)
y_pred = Best_Model.predict(x_test)
accuracy= accuracy_score(y_test, y_pred)
print(f'accuracy score: {accuracy}')
print(classification_report(y_test, y_pred, zero_division= 1))

# Include the confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

# plot with model name as title
disp.plot(cmap='Greens')
plt.title(f'confusion_matrix: {model_name}')
plt.show()

Hyperparater

In [ ]:
rfc= RandomForestClassifier(random_state=2)

rfc.get_params()


In [ ]:
# Grid of parameters to choose from
parameters = {
    "max_depth": np.arange(5, 16, 5),
    "min_samples_leaf": [3, 5, 7],
    "max_leaf_nodes": [2, 5],
    "min_impurity_decrease": [0.0001, 0.001],
}


In [ ]:
from sklearn.model_selection import GridSearchCV


rfc_obj = RandomForestClassifier()

grid_obj = GridSearchCV(rfc,param_grid=parameters,n_jobs=-1,verbose=True) ## Complete the code to run grid search with n_jobs = -1

grid_obj = grid_obj.fit(x_train,y_train) ## Complete the code to fit the grid_obj on the train data

# Set the clf to the best combination of parameters
rfc_estimator = grid_obj.best_estimator_



In [ ]:
# Fit the best algorithm to the data.
rfc_estimator.fit(x_train, y_train)

In [ ]:
# import joblib
# joblib.dump(model, 'model.pkl')
import joblib
from sklearn.metrics import classification_report
joblib.dump(Best_Model, "model.pkl")
# # save the scalar
joblib.dump(scaler, "scaler.pkl")

# print('model and scaler has been successful saved')

In [ ]:
y_pred = Best_Model.predict(x_test)
accuracy= accuracy_score(y_test, y_pred)
print(f'accuracy score: {accuracy}')
print(classification_report(y_test, y_pred, zero_division= 1))

In [ ]:
from sklearn import metrics
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import numpy as np

xgb = XGBClassifier(random_state=1, eval_metric="logloss")

parameters = {
    "n_estimators": np.arange(150, 250, 50),
    "scale_pos_weight": [1, 2],
    "subsample": [0.9, 1],
    "learning_rate": np.arange(0.1, 0.21, 0.1),
    "gamma": [3, 5],
    "colsample_bytree": [0.8, 0.9],
    "colsample_bylevel": [0.9, 1],
}

f1_scorer = metrics.make_scorer(metrics.f1_score)

grid_obj = GridSearchCV(xgb, param_grid=parameters, cv=5, verbose=True, scoring=f1_scorer)
grid_obj.fit(x_train, y_train)

best_xgb = grid_obj.best_estimator_

# best_xgb is already fitted on the whole training set
# Evaluate or predict with best_xgb

In [ ]:
# Evaluate the best XGBoost model from grid search
y_pred_xgb = best_xgb.predict(x_test)

# Accuracy and classification report
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("=== XGBoost Model Performance ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - XGBoost')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
#Cross Validation
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Use the best XGBoost model with its hyperparameters
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(best_xgb, x, y, cv=cv, scoring='accuracy')

print("3‑Fold Cross‑Validation Results:")
for i, score in enumerate(scores):
    print(f"Fold {i+1}: Accuracy = {score:.4f}")
print(f"Mean Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
# Define seeds and run three tests with confusion matrix plots

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define three different random seeds for three test splits
seeds = [1, 42, 123]

# Store results
test_results = []

for i, seed in enumerate(seeds):
    print(f"\n{'='*50}")
    print(f"Test {i+1} with random_state = {seed}")
    print('='*50)
    
    # Split data (use your original x, y from earlier preprocessing)
    x_tr, x_te, y_tr, y_te = train_test_split(
        x, y, test_size=0.3, random_state=seed, stratify=y
    )
    
    # Re‑train XGBoost with best hyperparameters found earlier
    from xgboost import XGBClassifier
    model = XGBClassifier(**best_xgb.get_params())
    model.fit(x_tr, y_tr)
    
    # Predict
    y_pred = model.predict(x_te)
    
    # Metrics
    acc = accuracy_score(y_te, y_pred)
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_te, y_pred))
    
    # Confusion matrix plot
    cm = confusion_matrix(y_te, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - Test {i+1} (seed={seed})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    test_results.append({
        'seed': seed,
        'accuracy': acc,
        'model': model,
        'confusion_matrix': cm
    })

# Summary of three tests
print("\n" + "="*50)
print("SUMMARY OF THREE TESTS")
print("="*50)
for res in test_results:
    print(f"Seed {res['seed']}: Accuracy = {res['accuracy']:.4f}")
mean_acc = np.mean([r['accuracy'] for r in test_results])
std_acc = np.std([r['accuracy'] for r in test_results])
print(f"\nMean accuracy: {mean_acc:.4f} (+/- {std_acc:.4f})")

In [ ]:
# (Optional) Baseline Logistic Regression on same three splits (with confusion matrices)

from sklearn.linear_model import LogisticRegression

baseline_results = []

for i, seed in enumerate(seeds):
    x_tr, x_te, y_tr, y_te = train_test_split(
        x, y, test_size=0.3, random_state=seed, stratify=y
    )
    lr = LogisticRegression(max_iter=1000)
    lr.fit(x_tr, y_tr)
    y_pred_lr = lr.predict(x_te)
    acc_lr = accuracy_score(y_te, y_pred_lr)
    baseline_results.append(acc_lr)
    
    # Confusion matrix for baseline
    cm_lr = confusion_matrix(y_te, y_pred_lr)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges')
    plt.title(f'Baseline Logistic Regression (seed={seed})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    print(f"Seed {seed}: Logistic Regression Accuracy = {acc_lr:.4f}")

print(f"\nBaseline mean accuracy: {np.mean(baseline_results):.4f}")
print(f"XGBoost mean accuracy: {np.mean([r['accuracy'] for r in test_results]):.4f}")

In [ ]:
#3‑fold cross‑validation as another sanity check

from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(best_xgb, x, y, cv=cv, scoring='accuracy')

print("3‑Fold Cross‑Validation Results:")
for i, score in enumerate(scores):
    print(f"Fold {i+1}: Accuracy = {score:.4f}")
print(f"Mean Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
# Plot feature importance
importances = best_xgb.feature_importances_
feature_names = x_train.columns
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10,6))
plt.title("Feature Importances (XGBoost)")
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.gca().invert_yaxis()
plt.show()

In [59]:
import joblib

# Save the best XGBoost model and scaler
joblib.dump(best_xgb, "best_xgb_model.pkl")
joblib.dump(scaler, "scaler.pkl")  # already saved earlier, but ensure it's the correct scaler

print("Model and scaler saved successfully.")

Model and scaler saved successfully.
